In [1]:
import aether.config as config
import numpy as np
import cupy as cp
import aether as ae

In [2]:
from pathlib import Path
# Navigate 2 levels up from 'examples/pure_grit/' to the repository root
REPO_ROOT = Path(__file__).resolve().parents[2] if "__file__" in locals() else Path.cwd().parents[1]
DATA_PATH = REPO_ROOT / "data" / "cifar-10-python.tar.gz"

print(REPO_ROOT)

print(f"\n{DATA_PATH}")

/app

/app/data/cifar-10-python.tar.gz


In [3]:
TRAIN_DIR = REPO_ROOT / "data" / "cifar-10" / "cifar-10_train.npz"
TEST_DIR  = REPO_ROOT / "data" / "cifar-10" / "cifar-10_test.npz"

with cp.load(TRAIN_DIR, allow_pickle=False) as data:
    X_train = data["X_train"]
    y_train = data["y_train"]

with cp.load(TEST_DIR, allow_pickle=False) as data:
    X_test = data["X_test"]
    y_test = data["y_test"]


Now lets do our preprocessing pipeline

In [4]:
TARGET_DEVICE = "cupy"
feature_pipeline = ae.Compose([
    ae.ToTensor(dtype='float32', target_device=TARGET_DEVICE),
    ae.Rescale(factor=1.0 / 255.0),
    ae.StandardScaler()
]).fit(X_train)

# 2. Transform train and test features seamlessly
X_train_tensor = feature_pipeline(X_train)
X_test_tensor = feature_pipeline(X_test)

# 3. Convert target labels
y_train_tensor, y_test_tensor = ae.to_tensor(
    y_train, y_test, target_device=TARGET_DEVICE, preserve_integers=True
)

print(f"{X_train.shape=}, {X_train.dtype=}, {type(X_train)=}")
print(f"{y_train.shape=}, {y_train.dtype=}, {type(y_train)}")
print(f"{X_train_tensor.shape=}, {X_train_tensor.dtype=}, {type(X_train_tensor)=}")
print(f"{y_train_tensor.shape=}, {y_train_tensor.dtype=}, {type(y_train_tensor)}")


X_train.shape=(50000, 32, 32, 3), X_train.dtype=dtype('uint8'), type(X_train)=<class 'cupy.ndarray'>
y_train.shape=(50000,), y_train.dtype=dtype('int64'), <class 'cupy.ndarray'>
X_train_tensor.shape=(50000, 32, 32, 3), X_train_tensor.dtype=dtype('float32'), type(X_train_tensor)=<class 'cupy.ndarray'>
y_train_tensor.shape=(50000,), y_train_tensor.dtype=dtype('int64'), <class 'cupy.ndarray'>


## Making a Simple CNN Model that Uses Half Precision

These models aren't meant to be serious contendors for benchmark performance, but rather a proof of concept.

In [6]:
model = ae.Model()
model.manual_seed(seed=42)

# --- Block 1: Low-level edges & colors (32x32 -> 16x16) ---
model.add(ae.Conv2d(3, 32, (3, 3), (1, 1), padding="same"))
model.add(ae.BatchNorm(epsilon=1e-5, momentum=0.9))
model.add(ae.ReLU())
model.add(ae.Conv2d(32, 32, (3, 3), (1, 1), padding="same"))
model.add(ae.BatchNorm(epsilon=1e-5, momentum=0.9))
model.add(ae.ReLU())
model.add(ae.MaxPool2d((2, 2), (2, 2), padding="valid"))
model.add(ae.SpatialDropout(rate=0.1, seed=42))

# --- Block 2: Intermediate textures & patterns (16x16 -> 8x8) ---
model.add(ae.Conv2d(32, 64, (3, 3), (1, 1), padding="same", l2=1e-5))
model.add(ae.BatchNorm(epsilon=1e-5, momentum=0.9))
model.add(ae.ReLU())
model.add(ae.Conv2d(64, 64, (3, 3), (1, 1), padding="same"))
model.add(ae.BatchNorm(epsilon=1e-5, momentum=0.9))
model.add(ae.ReLU())
model.add(ae.MaxPool2d((2, 2), (2, 2), padding="valid"))
model.add(ae.SpatialDropout(rate=0.15, seed=42))

# --- Block 3: High-level class semantics (8x8 -> 4x4) ---
model.add(ae.Conv2d(64, 128, (3, 3), (1, 1), padding="same"))
model.add(ae.BatchNorm(epsilon=1e-5, momentum=0.9))
model.add(ae.ReLU())
model.add(ae.MaxPool2d((2, 2), (2, 2), padding="valid"))
model.add(ae.SpatialDropout(rate=0.2, seed=42))

# --- Head: Parameter-efficient classification ---
model.add(ae.GlobalAvgPool())
model.add(ae.Dense(128, 10))

# --- Compilation & Training ---
model.configure(
    loss=ae.SoftmaxCategoricalCrossEntropy(label_smoothing=0.05),
    optimizer=ae.AdamW(lr=0.001, decay=1e-4, weight_decay=0.01),
    accuracy=ae.CategoricalAccuracy()
)

model.to('cupy')
model.set_precision(compute_dtype="float16")
model.finalize(input_shape=X_train_tensor.shape[1:])

model.train(
    X=X_train_tensor,
    y=y_train_tensor,
    epochs=25,
    batch_size=128,
    shuffle=True,
    print_every=100,
    verbose=1,
    validation_data=(X_test_tensor, y_test_tensor)
)

[██████████████████████████████] 100%  Step 391/391  122.3 it/s  3.2s | loss 1.4621 ▲0.0481 acc 52.50%  ▼5.31% reg 0.0007   lr 9.62e-04    
[Epoch 1/25 Total] loss: 1.7030 - acc: 41.61% - lr: 0.000962
[Validation] loss: 1.4399            acc: 47.08%            ★ new best acc
[██████████████████████████████] 100%  Step 391/391  173.9 it/s  2.2s | loss 1.2559 ▼0.0493 acc 60.00%  ▼0.94% reg 0.0007   lr 9.28e-04    
[Epoch 2/25 Total] loss: 1.3764 - acc: 54.95% - lr: 0.000928
[Validation] loss: 1.1001  ▼0.3398   acc: 61.39%  ▲14.31%   ★ new best acc
[██████████████████████████████] 100%  Step 391/391  173.2 it/s  2.3s | loss 1.1361 ▼0.0229 acc 68.75%  ▲4.69% reg 0.0007   lr 8.95e-04    
[Epoch 3/25 Total] loss: 1.2549 - acc: 60.89% - lr: 0.000895
[Validation] loss: 0.9733  ▼0.1269   acc: 66.30%   ▲4.91%   ★ new best acc
[██████████████████████████████] 100%  Step 391/391  172.9 it/s  2.3s | loss 1.0793 ▼0.0119 acc 68.75%  ▲2.34% reg 0.0007   lr 8.65e-04    
[Epoch 4/25 Total] loss: 1.1754 

In [8]:
model = ae.Model()
model.add(ae.Flatten())
model.add(ae.Dense(32*32*3, 256))
model.add(ae.ReLU())
model.add(ae.Dense(256, 128, l2=1e-5))
model.add(ae.Dense(128, 10))
model.configure(
    loss = ae.SoftmaxCategoricalCrossEntropy(label_smoothing=0.01),
    optimizer= ae.AdamW(learning_rate=0.001, decay=5e-5, weight_decay = 0.01),
    accuracy= ae.CategoricalAccuracy()
)
#model.set_precision("float16") works here too
model.to('cupy')
model.set_precision(compute_dtype="float16")
model.finalize(input_shape=X_train_tensor.shape[1:])
model.train(
    X=X_train_tensor, 
    y=y_train_tensor, 
    epochs=5, 
    batch_size=128,
    print_every=150, 
    validation_data=(X_test_tensor, y_test_tensor)
)

[██████████████████████████████] 100%  Step 391/391  1526.8 it/s  0.3s | loss 1.6105 ▼0.1514 acc 48.75% ▲ 8.12% reg 0.0012   lr 9.81e-04    
[Epoch 1/5 Total] loss: 1.9204 - acc: 0.4016 - lr: 0.000981
[Validation] loss 1.5618            acc 45.58%            ★ new best acc
[██████████████████████████████] 100%  Step 391/391  1560.1 it/s  0.3s | loss 1.5386 ▼0.3309 acc 46.25% ▲ 9.53% reg 0.0012   lr 9.62e-04    
[Epoch 2/5 Total] loss: 1.5352 - acc: 0.4761 - lr: 0.000962
[Validation] loss 1.5028  ▼0.0590   acc 48.14%  ▲ 2.56%   ★ new best acc
[██████████████████████████████] 100%  Step 391/391  1553.2 it/s  0.3s | loss 1.1482 ▼0.2464 acc 58.75% ▲10.31% reg 0.0012   lr 9.45e-04    
[Epoch 3/5 Total] loss: 1.4482 - acc: 0.5076 - lr: 0.000945
[Validation] loss 1.4951  ▼0.0078   acc 48.78%  ▲64.00%   ★ new best acc
[██████████████████████████████] 100%  Step 391/391  1572.6 it/s  0.2s | loss 1.5015 ▲0.2011 acc 57.50% ▲ 5.94% reg 0.0012   lr 9.28e-04    
[Epoch 4/5 Total] loss: 1.3956 - acc: